In [20]:
import httpx
import yaml
from httpx import HTTPStatusError
from phoenix.client import Client

from social_groups.directories import MULTIRUN_FINAL_RESULTS_DIR, RUNS_FINAL_RESULTS_DIR
from social_groups.trialrunner.utils.meta_info import ExperimentMetaInfo

In [21]:
client = Client(base_url="http://localhost:46001", http_client=httpx.Client(base_url="http://localhost:46001", timeout=60))

all_projects = client.projects.list()

In [22]:
important_projects = set()

for x in MULTIRUN_FINAL_RESULTS_DIR.iterdir():

    if not x.is_dir():
        continue
    for a in x.iterdir():
        if not a.is_dir():
            continue
        for b in a.iterdir():
            if not b.is_dir():
                continue
            for c in b.iterdir():
                if c.name == "meta.yaml":
                    important_projects.add(
                        ExperimentMetaInfo.model_validate(
                            yaml.safe_load(c.read_text())
                        ).phoenix_project_name
                    )

for x in RUNS_FINAL_RESULTS_DIR.iterdir():
    for a in x.iterdir():
        for b in a.iterdir():
            if b.name == "meta.yaml":
                important_projects.add(
                    ExperimentMetaInfo.model_validate(
                        yaml.safe_load(b.read_text())
                    ).phoenix_project_name
                )

important_projects

{'2026-04-18-08-43-04 - heterogeneous_group_baseline_0',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_1',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_10',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_11',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_2',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_3',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_4',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_5',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_6',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_7',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_8',
 '2026-04-18-08-43-04 - heterogeneous_group_baseline_9',
 '2026-04-18-08-43-31 - heterogeneous_group_baseline_structured_output_0',
 '2026-04-18-08-43-31 - heterogeneous_group_baseline_structured_output_1',
 '2026-04-18-08-43-31 - heterogeneous_group_baseline_structured_output_10',
 '2026-04-18-08-43-31 - heterogeneous_group_baseline_structured_output_11',
 '2026-04-18

In [23]:
all_projects

[{'name': '2026-06-13-12-00-01 - diversity_parameter_sweep_mad2_15',
  'description': None,
  'id': 'UHJvamVjdDo5MjE='},
 {'name': '2026-06-13-12-00-01 - diversity_parameter_sweep_mad2_9',
  'description': None,
  'id': 'UHJvamVjdDo5MjA='},
 {'name': '2026-06-13-12-00-01 - diversity_parameter_sweep_mad2_11',
  'description': None,
  'id': 'UHJvamVjdDo5MTk='},
 {'name': '2026-06-13-12-00-01 - diversity_parameter_sweep_mad2_10',
  'description': None,
  'id': 'UHJvamVjdDo5MTg='},
 {'name': '2026-06-13-12-00-01 - diversity_parameter_sweep_mad2_2',
  'description': None,
  'id': 'UHJvamVjdDo5MTc='},
 {'name': '2026-06-13-12-00-01 - diversity_parameter_sweep_mad2_4',
  'description': None,
  'id': 'UHJvamVjdDo5MTY='},
 {'name': '2026-06-13-12-00-01 - diversity_parameter_sweep_mad2_5',
  'description': None,
  'id': 'UHJvamVjdDo5MTU='},
 {'name': '2026-06-13-12-00-01 - diversity_parameter_sweep_mad2_8',
  'description': None,
  'id': 'UHJvamVjdDo5MTQ='},
 {'name': '2026-06-13-12-00-01 - dive

In [25]:
import datetime
to_delete = []

for project in all_projects:
    if project["name"] not in important_projects and project["name"] != "default":
        to_delete.append(project)


(to_delete
.sort(key=lambda x: datetime.datetime.strptime(x["name"].split(" - ")[0], "%Y-%m-%d-%H-%M-%S"), reverse=True)
)

to_delete = [x for x in to_delete if "2026-06-13-12" not in  x["name"]]
[x["name"] for x in to_delete]

['2026-06-09-09-42-13 - diversity_parameter_sweep_mad3_13',
 '2026-06-09-09-42-13 - diversity_parameter_sweep_mad3_11',
 '2026-06-09-09-42-13 - diversity_parameter_sweep_mad3_3',
 '2026-06-09-09-42-13 - diversity_parameter_sweep_mad3_1',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_11',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_14',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_2',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_10',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_4',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_5',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_8',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_15',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_3',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_9',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_1',
 '2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_13',
 '2026-06-09-08-54-53 - diversity_parameter_sweep

In [26]:
for project in to_delete:
    try:
        client.projects.delete(project_id=project["id"])
        print(f"Deleted {project['name']}")
    except HTTPStatusError as e:
        if e.response.status_code == 404:
            print(f" {project['name']} already deleted")
        else:
            print(f"Did NOT remove {project['name']}, because: ")
            print(e)


Deleted 2026-06-09-09-42-13 - diversity_parameter_sweep_mad3_13
Deleted 2026-06-09-09-42-13 - diversity_parameter_sweep_mad3_11
Deleted 2026-06-09-09-42-13 - diversity_parameter_sweep_mad3_3
Deleted 2026-06-09-09-42-13 - diversity_parameter_sweep_mad3_1
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_11
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_14
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_2
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_10
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_4
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_5
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_8
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_15
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_3
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_9
Deleted 2026-06-09-08-54-53 - diversity_parameter_sweep_mad2_1
Deleted 2026-06-09-08-54-53 - diversity_parameter